In [8]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from libpysal.weights import full2W
from esda.moran import Moran_Local
from PIL import Image
import os
from pyprojroot import here

In [9]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [10]:
gdf = gpd.read_file(RAW_DATA_DIR/"geographic/municipalities/00mun.shp")
lisa_results = pd.read_csv(RESULTS_DATA_DIR/"raw_esda_results/yearly_lisa_results.csv", encoding='latin-1')

In [11]:
lisa_results["CVEGEO"] = lisa_results["CVEGEO"].astype(str).str.zfill(5)  
gdf["CVEGEO"] = gdf["CVEGEO"].astype(str).str.zfill(5)  

In [12]:
CLUSTER_COLORS = {
    "HH": "#d7191c",   
    "LL": "#87c2ed",   
    "HL": "#f26b6d",   
    "LH": "#035fa3",   
    "ns": "#f0f0f0",   
    "island": "#ffffff",
}

years = sorted(lisa_results["year"].unique())

In [13]:
frame_paths = []
for year, year_df in lisa_results.groupby("year"):
    plot_gdf = gdf.merge(year_df[["CVEGEO", "cluster"]], on="CVEGEO", how="left")

    fig, ax = plt.subplots(figsize=(10, 10))
    for cluster_type, color in CLUSTER_COLORS.items():
        subset = plot_gdf[plot_gdf["cluster"] == cluster_type]
        subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)
    plot_gdf.boundary.plot(ax=ax, linewidth=0.1, color="#999999")
    ax.set_title(f"Crime Rate LISA Clusters — {year}", fontsize=16, fontweight="bold")
    ax.axis("off")
    legend_patches = [
        mpatches.Patch(color=CLUSTER_COLORS["HH"], label="High-High"),
        mpatches.Patch(color=CLUSTER_COLORS["LL"], label="Low-Low"),
        mpatches.Patch(color=CLUSTER_COLORS["HL"], label="High-Low (outlier)"),
        mpatches.Patch(color=CLUSTER_COLORS["LH"], label="Low-High (outlier)"),
        mpatches.Patch(color=CLUSTER_COLORS["ns"], label="Not significant"),
    ]
    ax.legend(handles=legend_patches, loc="lower left", fontsize=10, frameon=True)
    plt.savefig(RESULTS_DATA_DIR / f"raw_esda_results/LISA_{year}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    frame_paths.append(RESULTS_DATA_DIR / f"raw_esda_results/LISA_{year}.png")
    print(f"Saved frame for {year}")

C:\Users\franc\AppData\Local\Temp\ipykernel_88944\3921369919.py:8: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2015


C:\Users\franc\AppData\Local\Temp\ipykernel_88944\3921369919.py:8: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2016


C:\Users\franc\AppData\Local\Temp\ipykernel_88944\3921369919.py:8: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


...

Saved frame for 2023


C:\Users\franc\AppData\Local\Temp\ipykernel_88944\3921369919.py:8: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2024


C:\Users\franc\AppData\Local\Temp\ipykernel_88944\3921369919.py:8: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  subset.plot(ax=ax, color=color, edgecolor="none", linewidth=0)


Saved frame for 2025


In [14]:
frames = [Image.open(p) for p in frame_paths]
max_w = max(f.width for f in frames)
max_h = max(f.height for f in frames)
padded = []
for f in frames:
    bg = Image.new("RGB", (max_w, max_h), "white")
    bg.paste(f, ((max_w - f.width) // 2, (max_h - f.height) // 2))
    padded.append(bg)

gif_path = RESULTS_DATA_DIR/"raw_esda_results/yearly_lisa_clusters.gif"
os.makedirs(RESULTS_DATA_DIR/"raw_esda_results", exist_ok=True)
padded[0].save(
    gif_path,
    save_all=True,
    append_images=padded[1:],
    duration=1000,   # ms per frame
    loop=0,
)
print(f"GIF saved to {gif_path}")

GIF saved to C:\Users\franc\OneDrive\Documents\Repositories\organized-crime-prediction\results\raw_esda_results\yearly_lisa_clusters.gif
